In [1]:
import pandas as pd
import os
import numpy as np
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
# definindo os caminhos
path = os.getenv('DATASETS_PATH')
data_path = os.getenv('DATA_PATH')

# carregando dataset de players
df = pd.read_csv(path + '/players.csv')
df.head()

,player_id,first_name,last_name,name,last_season,current_club_id,player_code,country_of_birth,city_of_birth,country_of_citizenship,...,agent_name,image_url,international_caps,international_goals,current_national_team_id,url,current_club_domestic_competition_id,current_club_name,market_value_in_eur,highest_market_value_in_eur
0,10,Miroslav,Klose,Miroslav Klose,2015,398,miroslav-klose,Poland,Opole,Germany,...,ASBW Sport Marketing,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/miroslav-klose...,IT1,Società Sportiva Lazio S.p.A.,1000000.0,30000000.0
1,26,Roman,Weidenfeller,Roman Weidenfeller,2017,16,roman-weidenfeller,Germany,Diez,Germany,...,Neubauer 13 GmbH,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/roman-weidenfe...,L1,Borussia Dortmund,750000.0,8000000.0
2,65,Dimitar,Berbatov,Dimitar Berbatov,2015,1091,dimitar-berbatov,Bulgaria,Blagoevgrad,Bulgaria,...,CSKA-AS-23 Ltd.,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/dimitar-berbat...,GR1,Panthessalonikios Athlitikos Omilos Konstantin...,1000000.0,34500000.0
3,77,NaN,Lúcio,Lúcio,2012,506,lucio,Brazil,Brasília,Brazil,...,NaN,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/lucio/profil/s...,IT1,Juventus Football Club,200000.0,24500000.0
4,80,Tom,Starke,Tom Starke,2017,27,tom-starke,East Germany (GDR),Freital,Germany,...,IFM,https://img.a.transfermarkt.technology/portrai...,NaN,NaN,NaN,https://www.transfermarkt.co.uk/tom-starke/pro...,L1,FC Bayern München,100000.0,3000000.0


In [4]:
# removendo colunas de leakage e colunas irrelevantes para o modelo
cols_drop = [
    'market_value_in_eur', 'highest_market_value_in_eur',
    'first_name', 'last_name', 'player_code', 'country_of_birth',
    'city_of_birth', 'contract_expiration_date', 'agent_name',
    'image_url', 'url', 'current_club_domestic_competition_id',
    'current_club_name'
]
df.drop(columns=cols_drop, inplace=True)
df.head()

,player_id,name,last_season,current_club_id,country_of_citizenship,date_of_birth,sub_position,position,foot,height_in_cm,international_caps,international_goals,current_national_team_id
0,10,Miroslav Klose,2015,398,Germany,1978-06-09 00:00:00,Centre-Forward,Attack,right,184.0,NaN,NaN,NaN
1,26,Roman Weidenfeller,2017,16,Germany,1980-08-06 00:00:00,Goalkeeper,Goalkeeper,left,190.0,NaN,NaN,NaN
2,65,Dimitar Berbatov,2015,1091,Bulgaria,1981-01-30 00:00:00,Centre-Forward,Attack,NaN,NaN,NaN,NaN,NaN
3,77,Lúcio,2012,506,Brazil,1978-05-08 00:00:00,Centre-Back,Defender,NaN,NaN,NaN,NaN,NaN
4,80,Tom Starke,2017,27,Germany,1981-03-18 00:00:00,Goalkeeper,Goalkeeper,right,194.0,NaN,NaN,NaN


In [5]:
# parse da data de nascimento
df['date_of_birth'] = pd.to_datetime(df['date_of_birth'], errors='coerce')
df['date_of_birth'].dtype

dtype('<M8[ns]')

In [6]:
# preenchendo international_caps e international_goals nulos com 0
df['international_caps'] = df['international_caps'].fillna(0)
df['international_goals'] = df['international_goals'].fillna(0)
df[['international_caps', 'international_goals']].isna().sum()

international_caps     0
international_goals    0
dtype: int64

In [7]:
# criando national_team_ranking_inv sendo ranking FIFA invertido (top 1 = max pontos, sem seleção = 0)
df_temp = pd.read_csv(path + '/national_teams.csv', usecols=['national_team_id', 'fifa_ranking'])
nat_map = df_temp.set_index('national_team_id')['fifa_ranking'].to_dict()
max_ranking = df_temp['fifa_ranking'].max()

df['national_team_ranking_inv'] = df['current_national_team_id'].map(nat_map)
df['national_team_ranking_inv'] = df['national_team_ranking_inv'].apply(
    lambda r: int(max_ranking + 1 - r) if pd.notna(r) else 0
)
df.drop(columns=['current_national_team_id'], inplace=True)
df['national_team_ranking_inv'].value_counts(dropna=False).head()

national_team_ranking_inv
0      42965
169       96
184       89
205       82
186       76
Name: count, dtype: int64

In [8]:
df.head()

,player_id,name,last_season,current_club_id,country_of_citizenship,date_of_birth,sub_position,position,foot,height_in_cm,international_caps,international_goals,national_team_ranking_inv
0,10,Miroslav Klose,2015,398,Germany,1978-06-09,Centre-Forward,Attack,right,184.0,0.0,0.0,0
1,26,Roman Weidenfeller,2017,16,Germany,1980-08-06,Goalkeeper,Goalkeeper,left,190.0,0.0,0.0,0
2,65,Dimitar Berbatov,2015,1091,Bulgaria,1981-01-30,Centre-Forward,Attack,NaN,NaN,0.0,0.0,0
3,77,Lúcio,2012,506,Brazil,1978-05-08,Centre-Back,Defender,NaN,NaN,0.0,0.0,0
4,80,Tom Starke,2017,27,Germany,1981-03-18,Goalkeeper,Goalkeeper,right,194.0,0.0,0.0,0


In [9]:
# imputando height_in_cm com a mediana por position
height_median_by_position = df.groupby('position')['height_in_cm'].transform('median')
df['height_in_cm'] = df['height_in_cm'].fillna(height_median_by_position)
df['height_in_cm'].isna().sum()

0

In [10]:
# encoding ordinal de position
position_map = {
    'Goalkeeper': 1,
    'Defender':   2,
    'Midfield':   3,
    'Attack':     4,
    'Missing':    0
}
df['position_rank'] = df['position'].map(position_map).fillna(0).astype(int)
df.drop(columns=['position'], inplace=True)
df['position_rank'].value_counts()

position_rank
2    15414
3    13786
4    13064
1     5611
0      505
Name: count, dtype: int64

In [11]:
# OHE de sub_position
df = pd.get_dummies(df, columns=['sub_position'], prefix='sub_pos', dummy_na=False)
sub_pos_cols = [c for c in df.columns if c.startswith('sub_pos_')]
df[sub_pos_cols].sum()

sub_pos_Attacking Midfield    3386
sub_pos_Central Midfield      5288
sub_pos_Centre-Back           8452
sub_pos_Centre-Forward        6537
sub_pos_Defensive Midfield    3993
sub_pos_Goalkeeper            5611
sub_pos_Left Midfield          555
sub_pos_Left Winger           3176
sub_pos_Left-Back             3393
sub_pos_Right Midfield         564
sub_pos_Right Winger          3072
sub_pos_Right-Back            3569
sub_pos_Second Striker         279
dtype: int64

In [12]:
# OHE de foot, sendo right, left, both
df = pd.get_dummies(df, columns=['foot'], prefix='foot', dummy_na=False)
foot_cols = [c for c in df.columns if c.startswith('foot_')]
df[foot_cols].sum()

foot_both      1784
foot_left     10700
foot_right    30608
dtype: int64

In [13]:
# carregando countries para mapear country_of_citizenship para a confederação
df_countries = pd.read_csv(path + '/countries.csv')[['country_name', 'confederation']]
df_countries.head()

,country_name,confederation
0,Armenia,europa
1,North Macedonia,europa
2,Malaysia,asien
3,Maldives,asien
4,Malta,europa


In [14]:
# mapeando country_of_citizenship para confederação
confederation_map = df_countries.set_index('country_name')['confederation'].to_dict()
df['confederation'] = df['country_of_citizenship'].map(confederation_map)
df.drop(columns=['country_of_citizenship'], inplace=True)
df['confederation'].value_counts(dropna=False)

confederation
europa     29494
amerika     8303
NaN         4115
asien       3950
afrika      2518
Name: count, dtype: int64

In [15]:
# encoding ordinal de confederation, sendo europa=1, america=2, africa=3, asia=4, outros/nulo=0
conf_map = {
    'europa':  1,
    'amerika': 2,
    'afrika':  3,
    'asien':   4
}
df['confederation_encoded'] = df['confederation'].map(conf_map).fillna(0).astype(int)
df.drop(columns=['confederation'], inplace=True)
df['confederation_encoded'].value_counts()

confederation_encoded
1    29494
2     8303
0     4115
4     3950
3     2518
Name: count, dtype: int64

In [18]:
# verificando nulos restantes antes de salvar
df.isna().sum()[df.isna().sum() > 0]

Series([], dtype: int64)

In [17]:
# Remover jogadores sem date_of_birth
df = df.dropna(subset=['date_of_birth'])

In [19]:
# salvando players processado em parquet
df.to_parquet(data_path + '/players_processed.parquet', index=False)